# OneVoice V2 — Safety audio local

Sinh WAV safety dựng sẵn cho hai chiều và manifest SHA-256 trên Google Drive. Runtime edge chỉ đọc các WAV này; không tự tải hay tự tổng hợp audio. Cell có `--resume`, nên chạy lại sau khi Colab ngắt sẽ giữ WAV đã kiểm tra checksum.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
WORK_ROOT = Path('/content/drive/MyDrive/OneVoice')
# Team-approved traceability label. Change it only when the reviewed voice/phrases change.
APPROVAL_ID = 'impact-safety-v1'
OUTPUT_DIR = WORK_ROOT / 'artifacts/safety_audio_v1'
SAFETY_CSV = WORK_ROOT / 'review/safety_fast_path_review.csv'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'PyYAML', 'soundfile', 'gTTS', 'pydub'], check=True)
print('Source:', REPO)
print('Output:', OUTPUT_DIR)
if not SAFETY_CSV.is_file():
    raise FileNotFoundError(f'Missing reviewed safety CSV: {SAFETY_CSV}. Run colab_safety_review_v2.ipynb first.')
print('Approval ID:', APPROVAL_ID)


In [ ]:
command = [
    sys.executable, 'scripts/build_safety_audio.py',
    '--safety-csv', str(SAFETY_CSV),
    '--output', str(OUTPUT_DIR),
    '--profile', 'development',
    '--approval-id', APPROVAL_ID,
    '--resume',
]
print('> ' + ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('Safety-audio generation stopped. Run this same cell again: checksum-verified WAVs are resumed from Drive.')


In [ ]:
import json
from safety.audio_store import SafetyAudioStore

manifest = OUTPUT_DIR / 'manifest.json'
if not manifest.is_file():
    raise FileNotFoundError('Final manifest is absent; do not use partial audio in runtime.')
store = SafetyAudioStore(manifest, source_csv=REPO / 'data/onevoice_construction_v2/safety_fast_path.csv')
payload = json.loads(manifest.read_text(encoding='utf-8'))
print(f"Verified {len(payload['entries'])} safety WAV artifacts; approval={payload['approval_id']}")
display(payload)
